In [ ]:
#Install requirements
%pip install -r "../requirements.txt"

In [ ]:
#Import required libraries
import cv2
import numpy as np
from scipy import stats
from skimage import filters
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import os
import gc
import sys

In [ ]:
#Add the src to the path
sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
# BATCH PROCESSING FUNCTION


def batch_process_hierarchical(quantifier, root_dir, markers, regions,  groups,
                               threshold_method, threshold_param, registry, tissue_threshold=240): 
    """
    Process images in hierarchical folder structure and save results.
    
    Folder structure:
    root_dir/
        MARKER/
            REGION/
                GROUP/
                    SUBJECT/
                        DAB_RGB_Verification/
                            image1.jpg
                            image2.jpg
                            ...
                            Calculated_Masks_Final/  
                                mask_image1.jpg
                                mask_image2.jpg
                                ...
    
    Output CSVs:
    root_dir/MARKER/REGION_GROUP_Analysis_Results.csv
    """
    
    for marker in markers:
        for region in regions:
            for group in groups:
                group_path = os.path.join(root_dir, marker, 'Deconvolved_Results', region, group)
                if not os.path.exists(group_path):
                    print(f"  Skipping {group} (Path not found)")
                    continue

                all_data = []
                
                # Find subject folders
                subjects = sorted([
                    s for s in os.listdir(group_path) 
                    if os.path.isdir(os.path.join(group_path, s)) 
                    and s.startswith(('A', 'PDC'))
                ])
                
                if not subjects:
                    print(f"No subjects found in {group_path}")
                    continue
                
                print(f"Found {len(subjects)} subjects: {', '.join(subjects)}")
                
                for sub in subjects:
                    print(f"\n  Processing subject: {sub}")
                    
                    # 1. This is where the original files (with blue nuclei) are
                    input_folder_orig = os.path.join(root_dir, marker, region, group, sub)
                    
                    # 2. This is where your DAB-only images for quantification are
                    input_folder_dab = os.path.join(root_dir, marker, 'Deconvolved_Results', region, group, sub, 'DAB_RGB_Verification')
                    if not os.path.exists(input_folder_dab):
                        print(f"Missing DAB_RGB_Verification folder")
                        continue
                    
                    output_mask_folder = os.path.join(input_folder_dab, 'Calculated_Masks_Final')
                    os.makedirs(output_mask_folder, exist_ok=True)
                    
                    output_mask_folder = os.path.join(input_folder_dab, 'Calculated_Masks_Final')
                    

                    # Find image files
                    files = [
                        f for f in os.listdir(input_folder_dab) 
                        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff'))
                        and not f.startswith('mask_')  # Skip existing masks
                    ]
                    
                    if not files:
                        print(f"No images found")
                        continue
                    
                    print(f" Processing {len(files)} images...")

                    processed_count = 0
                    for filename in files:
                        img_path_dab = os.path.join(input_folder_dab, filename)
                        img_path_orig = os.path.join(input_folder_orig, filename)

                        current_key = f"{region}_{sub}_{filename}"

                        # Lookup the tissue count using the Triple Key
                        total_tissue = registry.get(current_key, 0)
                        if total_tissue == 0:
                            # Diagnostic print to help you find the mismatch
                            print(f"⚠️ Warning: Registry miss for {current_key}. Double check CSV spelling.")

                        # Quantify image
                        result = quantifier.quantify_image(
                            img_path_dab,
                            total_tissue,
                            threshold_method,
                            threshold_param
                        )

                        if result:
                            # Save mask
                            mask_path = os.path.join(output_mask_folder, f"mask_{filename}")
                            save_dual_source_mask(
                                img_path_orig=img_path_orig, 
                                img_path_dab=img_path_dab, 
                                result=result, 
                                output_path=mask_path, 
                                quantifier=quantifier, 
                                tissue_threshold=tissue_threshold
                            )
                            
                            # Store data for CSV
                            all_data.append({
                                'Image_Title': filename,
                                'Subject_ID': sub,
                                **result
                            })
                            
                            processed_count += 1
                    
                    print(f"Processed: {processed_count}/{len(files)} images")

                    # Clear memory
                    gc.collect()
                
                # Save CSV for this marker/region/group combination
                if all_data:
                    marker_path = os.path.join(root_dir, marker)
                    save_wide_format_csv(all_data, marker_path,  group)
                else:
                    print(f"No data collected for {group}")
    
    print(f"\n{'='*80}")
    print("✓ BATCH PROCESSING COMPLETE")
    print(f"{'='*80}\n")

def save_dual_source_mask(img_path_orig, img_path_dab, result, output_path, quantifier, tissue_threshold=240):
    """
    Final Correction: 
    - Uses Original RGB for Tissue Mask
    - Uses DAB Verification for Signal
    """
    # 1. Load Original for Tissue Masking
    orig_img = cv2.imread(str(img_path_orig))
    if orig_img is None: return
    
    # Grayscale thresholding on original catches nuclei as tissue
    gray_orig = cv2.cvtColor(orig_img, cv2.COLOR_BGR2GRAY)
    tissue_mask = gray_orig < tissue_threshold
    
    # 2. Load DAB Verification for Signal Detection
    dab_img = cv2.imread(str(img_path_dab))
    if dab_img is None: return
    
    # Convert DAB-only image to OD Space for thresholding
    dab_rgb = cv2.cvtColor(dab_img, cv2.COLOR_BGR2RGB)
    # Note: We use the same OD conversion logic as the main quantifier
    dab_od_img = -np.log((dab_rgb.astype(np.float32) / 255.0) + 1e-6)
    dab_signal = np.dot(dab_od_img, quantifier.dab_vector)
    
    # Subtract background as per your calibration
    dab_signal = np.clip(dab_signal - quantifier.reference_background, 0, None)
    
    # 3. Apply the threshold saved in 'result'
    threshold = result['threshold_value']
    dab_positive = (dab_signal > threshold) & tissue_mask
    
    # 4. Construct Visualization (Black Background)
    mask_vis = np.zeros((*dab_positive.shape, 3), dtype=np.uint8)
    
    # Layer 1: White Tissue (Nuclei-Aware)
    mask_vis[tissue_mask] = [255, 255, 255] 
    
    # Layer 2: Red Signal (DAB Isolated)
    mask_vis[dab_positive] = [255, 0, 0] 
    
    cv2.imwrite(output_path, cv2.cvtColor(mask_vis, cv2.COLOR_RGB2BGR))

def save_mask_visualization(original_img_path, result, output_path, quantifier, tissue_threshold=240):
    """
    Create and save mask visualization.
    White = tissue, Red = DAB positive, Black = background
    """
    orig_img = cv2.imread(str(original_img_path))
    if orig_img is None:
        return

    # 1.Recreate masks
    gray_orig = cv2.cvtColor(orig_img, cv2.COLOR_BGR2GRAY)
    tissue_mask = gray_orig < tissue_threshold 
    
    img_rgb = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)

    # 2.Extract DAB signal in OD space
    img_od = -np.log((img_rgb.astype(np.float32) / 255.0) + 1e-6)
    dab_od = np.dot(img_od, quantifier.dab_vector)
    dab_signal = np.clip(dab_od - quantifier.reference_background, 0, None)
    
    # 3.Apply the threshold from the result
    threshold = result['threshold_value']
    dab_positive = (dab_signal > threshold) & tissue_mask
    
    # 4.Construct the 3-layer visualization
    # Initialize as pure black (0, 0, 0)
    mask_vis = np.zeros((*dab_positive.shape, 3), dtype=np.uint8)
    
    # Layer 1: Paint the tissue area white
    mask_vis[tissue_mask] = [255, 255, 255]
    
    # Layer 2: Paint the DAB signal red
    mask_vis[dab_positive] = [255, 0, 0]
    
    # Save using BGR for OpenCV
    cv2.imwrite(output_path, cv2.cvtColor(mask_vis, cv2.COLOR_RGB2BGR))


def save_wide_format_csv(data_list, marker_path,  group):
    """
    Save results in wide format with each subject having its own tile column.
    Each subject gets 4 columns: Tile | Area_Percent | Mean_Intensity | DAB_Density
    """
    df_long = pd.DataFrame(data_list)
    
    # Get unique subjects
    subjects = sorted(df_long['Subject_ID'].unique())
    
    # Build dataframe for each subject
    subject_dfs = []
    max_tiles = 0
    
    for subject in subjects:
        # Filter data for this subject
        sub_data = df_long[df_long['Subject_ID'] == subject].copy()
        sub_data = sub_data.sort_values('Image_Title').reset_index(drop=True)
        
        # Create columns for this subject
        sub_df = pd.DataFrame({
            f'{subject}_Tile': sub_data['Image_Title'].values,
            f'{subject}_Area_Percent': sub_data['area_percent'].values,
            f'{subject}_Mean_Intensity': sub_data['mean_intensity'].values,
            f'{subject}_DAB_Density': sub_data['dab_density'].values
        })
        
        subject_dfs.append(sub_df)
        max_tiles = max(max_tiles, len(sub_df))
    
    # Pad dataframes to same length 
    for i in range(len(subject_dfs)):
        if len(subject_dfs[i]) < max_tiles:
            padding = max_tiles - len(subject_dfs[i])
            pad_df = pd.DataFrame(
                [[None] * len(subject_dfs[i].columns)] * padding,
                columns=subject_dfs[i].columns
            )
            subject_dfs[i] = pd.concat([subject_dfs[i], pad_df], ignore_index=True)
    
    # Concatenate all subjects horizontally
    df_wide = pd.concat(subject_dfs, axis=1)
    
    # Save to CSV
    output_csv = os.path.join(marker_path, f"{group}_Analysis_Results.csv")
    df_wide.to_csv(output_csv, index=False)
    
    print(f"\n CSV saved: {output_csv}")
    print(f"Subjects: {len(subjects)}")
    print(f"Max tiles per subject: {max_tiles}")
    print(f"Columns: {len(subjects)} subjects × 4 = {len(df_wide.columns)} columns")

In [ ]:
# CONFIGURATION

#Import function
from src.functions import DABQuantifier

ROOT_DIR = r'path_root' #Adjust the path root as needed
MARKERS = ['AQP4', 'GFAP', 'ABETA42'] #Adjust as needed
REGIONS = ['GM', 'WM'] #Adjust as needed
GROUPS = ['Controls','Patients'] #Adjust as needed

# Registry from Step 1
REGISTRY_FILE = r'tissue_registry.csv'

# Threshold settings
THRESHOLD_METHOD = 'fixed_adaptive'
THRESHOLD_PARAM = 3.5  
TISSUE_MASK_THRESHOLD = 240

# Calibration file
CALIBRATION_FILE = r'../data/background_calibration.npz'

In [ ]:
# MAIN EXECUTION

if __name__ == "__main__":
    
    print("\n" + "="*80)
    print("DAB QUANTIFICATION - BATCH PROCESSING")
    print("="*80)
    
    # 1. Load the Tissue Registry (The Denominator)
    print("\n1. Loading Tissue Registry...")
    try:
        registry_df = pd.read_csv(REGISTRY_FILE, sep=',')
        registry_df.columns = registry_df.columns.str.strip()

        # Create a Triple-Composite Key: "Region_SubjectID_ImageName"
        registry = {}
        for _, row in registry_df.iterrows():
            # Using string conversion to ensure no issues with data types
            triple_key = f"{row['Region']}_{row['Subject_ID']}_{row['Image_Name']}"
            registry[triple_key] = row['Total_Tissue_Pixels']

        print(f"✓ Registry loaded: {len(registry)} unique regional pairs identified.")
    except Exception as e:
        print(f"✗ ERROR loading registry: {e}")
        exit(1)

    # 2. Initialize quantifier and load calibration
    print("\n2. Loading calibration...")
    quantifier = DABQuantifier()
    try:
        quantifier.load_calibration(CALIBRATION_FILE)
    except FileNotFoundError:
        print(f"\n✗ ERROR: Calibration file not found: {CALIBRATION_FILE}")
        exit(1)
    
    # 3. Confirm Settings
    print(f"\n3. Ready to process {len(MARKERS)} markers across {len(GROUPS)} groups.")
    response = input("Proceed with batch processing? (yes/no): ")
    if response.lower() not in ['yes', 'y']:
        print("Aborted.")
        exit(0)
    
    # 4. Run the Hierarchical Pipeline
    # Note: We pass 'registry' and 'TISSUE_MASK_THRESHOLD' (for the masks)
    batch_process_hierarchical(
        quantifier=quantifier,
        root_dir=ROOT_DIR,
        markers=MARKERS,
        regions=REGIONS,
        groups=GROUPS,
        threshold_method=THRESHOLD_METHOD,
        threshold_param=THRESHOLD_PARAM,
        registry=registry,
        tissue_threshold=TISSUE_MASK_THRESHOLD
    )
    
    print("\n" + "="*80)
    print("✓ ALL PROCESSING COMPLETE!")
    print("="*80)
    